<a href="https://colab.research.google.com/github/dbalsx/26-1-OSS-Project-team-19/blob/test_v1/%EC%9D%B8%EB%8F%84%EB%B3%B4%ED%96%89_%EC%98%81%EC%83%81_%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%85%8B_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 깃허브 저장소 복제
!git clone https://github.com/ChelseaGH/sidewalk_prototype_AI_Hub.git

# 2. 작업 폴더로 이동
%cd sidewalk_prototype_AI_Hub

# 3. YOLO 등 객체 인식에 필요한 기본 라이브러리 설치
!pip install ultralytics opencv-python

Cloning into 'sidewalk_prototype_AI_Hub'...
remote: Enumerating objects: 1201, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 1201 (delta 0), reused 0 (delta 0), pack-reused 1199 (from 1)
Receiving objects: 100% (1201/1201), 47.26 MiB | 16.48 MiB/s, done.
Resolving deltas: 100% (247/247), done.
/content/sidewalk_prototype_AI_Hub
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/2019-01-004.인도보행영상_sample.zip' -d '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test'

Archive:  /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/2019-01-004.인도보행영상_sample.zip
   creating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/bbox_sample.xml  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027451.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027452.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027453.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027454.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027455.jpg  
  inflating: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/MP_SEL_B027456.jpg  
  inflating: /content/drive/MyDrive

In [ ]:
!mv /content/sidewalk_prototype_AI_Hub '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/'

In [ ]:
import os
import xml.etree.ElementTree as ET

# 1. 경로 설정 (1개짜리 통짜 XML 파일 경로로 직접 지정)
xml_file = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/bbox/bbox_sample.xml'
output_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/yolo_labels'

os.makedirs(output_dir, exist_ok=True)
classes = []

# 정규화 수학 공식 (그대로 사용)
def convert_coordinates(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x_center = (box[0] + box[1]) / 2.0
    y_center = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    x_center = x_center * dw
    w = w * dw
    y_center = y_center * dh
    h = h * dh
    return (x_center, y_center, w, h)

print("단일 XML 파일 파싱을 시작합니다...")

# XML 파일 읽기
tree = ET.parse(xml_file)
root = tree.getroot()

# 2. <image> 태그들을 하나씩 돌면서 처리
image_count = 0
for image_tag in root.iter('image'):
    # 태그의 '속성(get)'에서 이름과 사이즈 가져오기
    image_name = image_tag.get('name')
    w_image = float(image_tag.get('width'))
    h_image = float(image_tag.get('height'))

    # 확장자(.jpg) 떼고 txt 파일명 만들기
    filename_without_ext = os.path.splitext(image_name)[0]
    txt_path = os.path.join(output_dir, filename_without_ext + '.txt')

    # 해당 이미지 안에 있는 장애물 <box> 찾기
    boxes = list(image_tag.iter('box'))

    # 장애물이 1개라도 있는 이미지일 경우에만 txt 파일 생성
    if len(boxes) > 0:
        image_count += 1
        with open(txt_path, 'w') as out_file:
            for box in boxes:
                cls_name = box.get('label')

                if cls_name not in classes:
                    classes.append(cls_name)
                cls_id = classes.index(cls_name)

                # AI 허브의 특이한 좌표 이름(xtl, xbr, ytl, ybr)에서 값 가져오기
                b = (float(box.get('xtl')), float(box.get('xbr')),
                     float(box.get('ytl')), float(box.get('ybr')))

                bb = convert_coordinates((w_image, h_image), b)
                out_file.write(str(cls_id) + " " + " ".join([str(a) for a in bb]) + '\n')

# 3. classes.txt 만들기
classes_file = os.path.join(output_dir, 'classes.txt')
with open(classes_file, 'w') as f:
    for cls in classes:
        f.write(cls + '\n')

print(f"변환 완료! 총 {image_count}개의 이미지에 대한 TXT 라벨이 생성되었습니다.")
print(f"총 {len(classes)}개의 장애물 클래스가 식별되었습니다.")

단일 XML 파일 파싱을 시작합니다...
변환 완료! 총 299개의 이미지에 대한 TXT 라벨이 생성되었습니다.
총 25개의 장애물 클래스가 식별되었습니다.


In [ ]:
import yaml
import os
import shutil
import glob

# 1. 경로 세팅
base_dir = '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test'
image_dir = os.path.join(base_dir, 'bbox')
label_dir = os.path.join(base_dir, 'yolo_labels')
yaml_path = os.path.join(base_dir, 'data.yaml')

# 2. YOLO가 라벨을 잘 찾게 하려면 이미지와 txt 파일이 같은 폴더에 있는 것이 제일 확실해!
# yolo_labels 폴더에 생성된 txt 파일들을 이미지(bbox) 폴더로 모두 복사
txt_files = glob.glob(os.path.join(label_dir, '*.txt'))
for txt in txt_files:
    shutil.copy(txt, image_dir)
print("텍스트 라벨 파일들을 이미지 폴더로 성공적으로 합쳤습니다.")

# 3. classes.txt 읽어와서 클래스 목록 만들기
classes_file = os.path.join(label_dir, 'classes.txt')
with open(classes_file, 'r', encoding='utf-8') as f:
    # 빈 줄 제외하고 클래스 이름만 깔끔하게 리스트로 저장
    classes = [line.strip() for line in f.readlines() if line.strip()]

# 4. 우리가 원했던 data.yaml 내용 구성하기
data = {
    'train': image_dir,  # 학습용 이미지 경로
    'val': image_dir,    # 검증용 이미지 경로 (샘플이라 일단 동일하게!)
    'nc': len(classes),  # 클래스 총 개수
    'names': classes     # 클래스 이름 목록
}

# 5. data.yaml 파일로 예쁘게 저장
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, default_flow_style=None, allow_unicode=True)

print(f"data.yaml 생성 완료! 저장 위치: {yaml_path}")

텍스트 라벨 파일들을 이미지 폴더로 성공적으로 합쳤습니다.
data.yaml 생성 완료! 저장 위치: /content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/data.yaml


In [ ]:
%cd '/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝'

/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝


In [ ]:
!pip install ultralytics

In [ ]:
!yolo task=detect mode=train model=yolov8n.pt data='/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/data.yaml' epochs=10 imgsz=640 batch=16 project='/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/runs' name='sidewalk_sample_train'

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/26-1 오픈소스프로그래밍 팀프젝/dataset_test/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, h